# Phân loại ảnh thời trang bằng CNN (Fashion-MNIST)

So sánh **CNN** vs **MLP** trên bộ dữ liệu Fashion-MNIST, áp dụng **Data Augmentation**, **Dropout**, **Batch Normalization**.
Đánh giá bằng **Accuracy, Precision, Recall, F1-score, Confusion Matrix**.

> Chạy được trực tiếp trên **Google Colab**: mở notebook này trên Colab, chọn Runtime > Change runtime type > GPU, rồi chạy tuần tự từ trên xuống dưới. Notebook tự cài đặt thư viện cần thiết và tự tải Fashion-MNIST.


## 1. Setup môi trường

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "torch", "torchvision", "scikit-learn", "seaborn", "matplotlib", "tqdm"])
    print("Colab detected: dependencies installed.")
else:
    print("Local environment detected: assuming requirements.txt already installed.")


In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

DATA_DIR = Path("data")
FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]


## 2. Tải dữ liệu & Data Augmentation

Data augmentation (chỉ áp dụng cho tập train): lật ngang, xoay nhẹ, tịnh tiến nhẹ — giúp mô hình tổng quát hóa tốt hơn, giảm overfitting.

In [ ]:
MEAN, STD = (0.2860,), (0.3530,)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

BATCH_SIZE = 128
VAL_SPLIT = 0.1

full_train_aug = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=train_transform)
full_train_eval = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=eval_transform)
test_set = datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=eval_transform)

n_val = int(len(full_train_aug) * VAL_SPLIT)
n_train = len(full_train_aug) - n_val
generator = torch.Generator().manual_seed(SEED)
train_idx, val_idx = random_split(range(len(full_train_aug)), [n_train, n_val], generator=generator)

train_set = Subset(full_train_aug, train_idx.indices)
val_set = Subset(full_train_eval, val_idx.indices)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")


### Xem thử một số ảnh mẫu

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
raw_set = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True)
for ax in axes.flat:
    idx = random.randrange(len(raw_set))
    img, label = raw_set[idx]
    ax.imshow(img, cmap="gray")
    ax.set_title(CLASS_NAMES[label], fontsize=8)
    ax.axis("off")
fig.tight_layout()
plt.show()


## 3. Định nghĩa mô hình

- **MLP**: baseline fully-connected, có BatchNorm + Dropout.
- **CNN**: 2 khối Conv (Conv-BN-ReLU x2 -> MaxPool -> Dropout), sau đó fully-connected có BatchNorm + Dropout.

In [ ]:
class MLP(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class CNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(dropout / 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(dropout / 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


## 4. Vòng lặp huấn luyện / đánh giá

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


def fit(model, train_loader, val_loader, epochs=20, lr=1e-3, weight_decay=1e-4):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    progress = tqdm(range(1, epochs + 1), desc="Epochs")
    for epoch in progress:
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        progress.set_postfix(
            train_loss=f"{train_loss:.4f}", train_acc=f"{train_acc:.4f}",
            val_loss=f"{val_loss:.4f}", val_acc=f"{val_acc:.4f}",
        )
    return history


@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(DEVICE)
        preds = model(images).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


def plot_history(history, title, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title(f"{title} - Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="val")
    axes[1].set_title(f"{title} - Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_confusion(y_true, y_pred, title, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


## 5. Huấn luyện mô hình MLP (baseline)

In [ ]:
EPOCHS = 20

mlp_model = MLP(dropout=0.3)
mlp_history = fit(mlp_model, train_loader, val_loader, epochs=EPOCHS)
plot_history(mlp_history, "MLP", save_path=FIGURE_DIR / "mlp_history.png")


## 6. Huấn luyện mô hình CNN

In [ ]:
cnn_model = CNN(dropout=0.3)
cnn_history = fit(cnn_model, train_loader, val_loader, epochs=EPOCHS)
plot_history(cnn_history, "CNN", save_path=FIGURE_DIR / "cnn_history.png")


## 7. Đánh giá trên tập test: Accuracy, Precision, Recall, F1-score, Confusion Matrix

In [ ]:
mlp_pred, mlp_true = predict(mlp_model, test_loader)
print("=== MLP classification report ===")
mlp_report = classification_report(mlp_true, mlp_pred, target_names=CLASS_NAMES, digits=4, output_dict=True)
print(classification_report(mlp_true, mlp_pred, target_names=CLASS_NAMES, digits=4))
plot_confusion(mlp_true, mlp_pred, "MLP Confusion Matrix", save_path=FIGURE_DIR / "mlp_confusion_matrix.png")


In [ ]:
cnn_pred, cnn_true = predict(cnn_model, test_loader)
print("=== CNN classification report ===")
cnn_report = classification_report(cnn_true, cnn_pred, target_names=CLASS_NAMES, digits=4, output_dict=True)
print(classification_report(cnn_true, cnn_pred, target_names=CLASS_NAMES, digits=4))
plot_confusion(cnn_true, cnn_pred, "CNN Confusion Matrix", save_path=FIGURE_DIR / "cnn_confusion_matrix.png")


## 8. So sánh CNN vs MLP

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["MLP", "CNN"],
    "Accuracy": [mlp_report["accuracy"], cnn_report["accuracy"]],
    "Precision (macro)": [mlp_report["macro avg"]["precision"], cnn_report["macro avg"]["precision"]],
    "Recall (macro)": [mlp_report["macro avg"]["recall"], cnn_report["macro avg"]["recall"]],
    "F1-score (macro)": [mlp_report["macro avg"]["f1-score"], cnn_report["macro avg"]["f1-score"]],
    "Params": [sum(p.numel() for p in mlp_model.parameters()), sum(p.numel() for p in cnn_model.parameters())],
})
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mlp_history["val_acc"], label="MLP val_acc")
ax.plot(cnn_history["val_acc"], label="CNN val_acc")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Accuracy")
ax.set_title("MLP vs CNN - Validation Accuracy qua các epoch")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mlp_vs_cnn_val_acc.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Lưu mô hình lên Google Drive

File này là notebook **gốc — nơi duy nhất dùng để train**. Sau khi train xong, checkpoint (trọng số + lịch sử huấn luyện) sẽ được lưu lên Google Drive tại `MyDrive/BTL_AI_checkpoints/`. Hai notebook `Fashion_MNIST_Experiments.ipynb` và `Fashion_MNIST_Demo.ipynb` **chỉ đọc lại** checkpoint này, không tự train.

Khi chạy cell dưới, Colab sẽ hỏi xin quyền truy cập Google Drive — bấm **Allow** để tiếp tục.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CKPT_DIR = Path("/content/drive/MyDrive/BTL_AI_checkpoints")
except ImportError:
    CKPT_DIR = Path("checkpoints")  # không chạy trên Colab: lưu local

CKPT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(mlp_model.state_dict(), CKPT_DIR / "mlp.pt")
torch.save(cnn_model.state_dict(), CKPT_DIR / "cnn.pt")
with open(CKPT_DIR / "mlp_history.json", "w") as f:
    json.dump(mlp_history, f)
with open(CKPT_DIR / "cnn_history.json", "w") as f:
    json.dump(cnn_history, f)

print("Saved checkpoints to", CKPT_DIR.resolve())


## 10. Kết luận

Điền nhận xét dựa trên bảng so sánh và các biểu đồ ở trên, ví dụ:
- CNN có Accuracy/F1-score cao hơn MLP hay không, chênh lệch bao nhiêu?
- Data Augmentation, Dropout, Batch Normalization ảnh hưởng thế nào đến khoảng cách giữa train và val accuracy (overfitting)?
- Các lớp nào (class) bị nhầm lẫn nhiều nhất qua confusion matrix (ví dụ Shirt vs T-shirt/top vs Pullover vs Coat)?
